# 🐍 Master Guide: Multiprocessing & True Parallelism
> **Single-Source Revision Guide**: Covers bypassing the GIL, CPU-bound workloads, process pools (`Pool.map`), Inter-Process Communication (IPC: `Queue`, `Pipe`), and Windows spawn pitfalls.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Unlike threads which share a single Python interpreter and GIL, **processes run independent Python interpreters across multiple CPU cores**, achieving true parallel execution for heavy CPU-bound computing.

---
## 🧠 2. CPython Architecture: Fork vs Spawn & IPC

### Fork vs Spawn Start Methods
- **`spawn` (Windows & macOS default)**: Starts a fresh Python process from scratch. Re-imports the entry script (which is why `if __name__ == '__main__':` is MANDATORY on Windows!).
- **`fork` (Legacy Linux)**: Clones the parent process memory. Fast, but unsafe in multi-threaded programs.

### Inter-Process Communication (IPC)
Processes do NOT share memory. Objects passed between processes must be serialized via `pickle` across OS pipes or shared memory buffers.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Parallel Mapping with `multiprocessing.Pool`

In [ ]:
from multiprocessing import Pool, cpu_count

def heavy_computation(x):
    return sum(i * i for i in range(x))

print(f"Available CPU Cores: {cpu_count()}")

# Note: In Jupyter / Windows scripts, multiprocessing works cleanly with Pool:
numbers = [100_000, 200_000, 300_000, 400_000]

# Using Pool to parallelize across available cores:
with Pool(processes=2) as pool:
    results = pool.map(heavy_computation, numbers)

print("Computation results:", results)

---
## 🟡 4. Part 2: Intermediate Features

### Inter-Process Queues (`multiprocessing.Queue`)
Process-safe and thread-safe queue for streaming messages between worker processes.

In [ ]:
from multiprocessing import Process, Queue

def producer(q):
    for item in ["item_A", "item_B", "item_C"]:
        q.put(item)

q = Queue()
p = Process(target=producer, args=(q,))
p.start()
p.join()

items = []
while not q.empty():
    items.append(q.get())
print("Items received from child process:", items)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Windows Spawn Infinite Process Fork Bomb
On Windows, new processes re-import your script. If code creating processes is not protected by `if __name__ == '__main__':`, each child process spawns more child processes indefinitely until the system crashes!

In [ ]:
print("Rule: ALWAYS guard multiprocessing process creation with:")
print("  if __name__ == '__main__':")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| multiprocessing                    | Bypasses the GIL; true multicore speed|
| Best for                           | CPU-bound heavy calculations          |
| Memory                             | Separate memory spaces (needs IPC)    |
| IPC Mechanisms                     | Queue, Pipe, Value, Array             |
| if __name__ == '__main__':         | MANDATORY on Windows and macOS (spawn)|
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. CPU-Bound Batch Processing & Worker Pool Scaling
> **Core Principle**:
> - For CPU-bound operations (numerical analysis, cryptographic hashing, image processing), threads cannot utilize multiple CPU cores due to the GIL.
> - `multiprocessing.Pool` distributes tasks across separate OS processes, achieving true linear multi-core speedup.
> - `pool.map()` blocks until all results complete; `pool.imap()` yields results lazily.

In [ ]:
# Practical Implementation: CPU-Bound Worker Pool
import multiprocessing as mp
import time

def compute_square_sum(n):
    """Simulates CPU-heavy mathematical batch calculation."""
    return sum(i * i for i in range(1, n + 1))

batch_inputs = [500_000, 1_000_000, 1_500_000, 2_000_000]

print("Starting parallel batch execution with multiprocessing Pool:")
start_t = time.perf_counter()

with mp.Pool(processes=min(4, mp.cpu_count())) as pool:
    batch_results = pool.map(compute_square_sum, batch_inputs)

elapsed = time.perf_counter() - start_t
print(f"Completed {len(batch_results)} heavy tasks in {elapsed*1000:.2f} ms")
for inp, res in zip(batch_inputs, batch_results):
    print(f"  compute_square_sum({inp:,}) -> {res}")

### 2. Multi-Process Pipeline: Producer-Consumer with `mp.Queue` & Sentinel Shutdown
Clean inter-process message streaming where producer and consumer run in isolated processes.

In [ ]:
# Multi-Process Pipeline with Sentinel Termination
import multiprocessing as mp

def data_producer(task_queue, raw_records):
    """Generates data items and sends them through the queue."""
    for record in raw_records:
        task_queue.put(record)
    # Send None sentinel to indicate work is done
    task_queue.put(None)

def data_consumer(task_queue, result_list):
    """Consumes items until sentinel is received."""
    while True:
        item = task_queue.get()
        if item is None:
            break
        result_list.append(item.upper())

channel = mp.Queue()
test_data = ["alpha", "beta", "gamma", "delta"]
processed = []

prod_proc = mp.Process(target=data_producer, args=(channel, test_data))
prod_proc.start()
prod_proc.join()

data_consumer(channel, processed)
print("Pipeline output received from inter-process queue:", processed)